# 03. 학교 밖 접속 — Tailscale Funnel

| 항목 | 내용 |
|---|---|
| 목표 | 사설 IP·공인 IP 를 숨긴 채 사이트(HTTPS)만 인터넷에 공개 |
| 소요 시간 | 20분 |
| 시작 전 조건 | 02 완료, Tailscale 계정(무료) |
| 실행 위치 | 🖥 서버 콘솔 (절마다 다시 표시) |

> 대문자 대괄호 `[LIKE_THIS]` 는 **자기 환경의 값으로 바꿔 쓰는 자리표시자**입니다. 표는 [00_overview](00_overview.ipynb)에 있습니다.
> 이 노트북의 셀은 **문서**입니다. 설명대로 해당 위치(서버 콘솔 등)에서 직접 실행하세요. 명령에 따라 서버 설정이 바뀝니다.

## 왜 Funnel 인가
- 관리 서버가 Tailscale 로 **밖으로 터널을 걸고**, 사이트(HTTPS)만 공개합니다. **인바운드 포트를 하나도 열지 않습니다.**
- 주소 `https://[MGMT_NAME].[TAILNET].ts.net` — 교내·교외가 같고, 인증서는 Tailscale 이 자동 발급합니다(도메인 구매 불필요).
- 제약: Funnel 은 **HTTPS 만** 공개합니다 → 학교 밖 학생은 cmd SSH 가 안 되고 **웹 터미널**을 씁니다. 대용량 전송은 느릴 수 있습니다.

## A. 로그인 (🖥 서버 콘솔 + 🌐 브라우저)
1. https://tailscale.com 에서 계정을 만듭니다(관리자 본인 계정, 무료).
2. 서버에서 아래를 실행하면 `https://login.tailscale.com/a/…` 링크가 나옵니다. 링크가 길면 `--qr` 옵션으로 QR 을 띄워 휴대폰으로 찍으세요.
3. 브라우저에서 링크를 열어 로그인·승인하면 서버에 `Success.` 가 나옵니다.

In [ ]:
!sudo tailscale up --hostname [MGMT_NAME]
!tailscale ip -4                      # 100.x.x.x 가 나오면 성공

## B. 관리 콘솔 설정 (🌐 https://login.tailscale.com/admin)

1. **DNS** 탭 → **MagicDNS** 켜기, **HTTPS Certificates** 의 *Enable HTTPS* 누르기.
2. **Access controls** 탭 → 편집기에서 `nodeAttrs` 항목에 Funnel 허용을 넣고 **Save**. 이미 `nodeAttrs` 가 있으면 안의 `[ ]` 에 `{ … }` 한 줄만 추가합니다.

```json
"nodeAttrs": [
  { "target": ["autogroup:member"], "attr": ["funnel"] }
]
```

## C. 공개 (🖥 서버 콘솔)

사이트(K3s 의 웹 입구는 서버의 80 번)를 HTTPS 로 공개합니다. 처음에는 "Funnel is not enabled" 링크가 나올 수 있습니다 → 브라우저로 열어 승인한 뒤 다시 실행하세요.

In [ ]:
!sudo tailscale funnel --bg 80
!tailscale funnel status              # https://[MGMT_NAME].[TAILNET].ts.net 주소 확인

## D. 주소를 설정에 반영

확인한 주소를 저장소의 운영 값 파일(`deploy/prod/values-prod.yaml`)의 `config.publicUrl` 에 넣고 커밋·푸시합니다. (다음 노트북에서 배포되면 ArgoCD 가 반영합니다.)

```yaml
config:
  publicUrl: [FUNNEL_URL]
  mgmtPrivateIp: [MGMT_PRIVATE_IP]
  capstone:
    publicHost: [MGMT_PRIVATE_IP]
  litellm:
    studentUrl: http://[MGMT_PRIVATE_IP]:30400
```

> 사이트가 아직 배포 전이라면 이 주소를 열어도 `404 page not found` 가 보이는 것이 정상입니다. **휴대폰 데이터(교외망)로도** 열어 보세요.

## E. 접속자 IP 헤더 (나중에 확인)
사이트는 로그인 시도 제한·감사 로그에 접속자 IP 를 씁니다. Funnel 이 원래 IP 를 어떤 헤더로 주는지는 **설치 후 직접 확인**해야 하며, 확인 전에는 설정값 `clientIpHeader` 를 비워 둡니다(아이디별 제한은 영향 없음).

## 문제가 생기면
| 증상 | 조치 |
|---|---|
| `tailscale: command not found` | 02 의 설치 단계가 끝까지 안 됨 → Tailscale 설치 스크립트 실행 |
| 외부에서 안 열림 | `tailscale status`, `tailscale netcheck`(학교가 UDP 41641·DERP 443 을 막는지) |
| 서버 **자신**에서 자기 주소가 `self-signed certificate` | 정상. 06 의 "관리 서버 자신의 가입" 우회 참고 |

## ✅ 완료 확인

- [ ] `tailscale ip -4` 가 100.x 주소를 출력
- [ ] `tailscale funnel status` 에 공개 주소가 있다
- [ ] 휴대폰 데이터로 주소가 열린다(404 라도 연결은 됨)